# QKeras High/Low Spatial Frequency Decoupling with QOctaveConv2D

This notebook demonstrates how to build a Quantization-Aware Training (QAT) structure utilizing **Octave Convolutions (QOctaveConv2D)** via QKeras. This topology splits image tensor spaces into multi-frequency streams for computing efficiency.

### Core Architectural Concepts Shown in This Example

* **Dual-Frequency Feature Mapping (`alpha`):** Standard convolutions process feature representations uniformly across fixed spatial shapes. Octave Convolutions separate smooth, low-frequency base shapes (textures, blurry illumination planes) from sharp high-frequency changes (fine edges, micro-details). The `alpha` ratio parameter isolates a subset of target feature channels, downscaling their dimensional surface area by half ($1/2$). Dropping this resolution dramatically decreases global execution storage requirements.
* **Separable Optimization Over Quantized Spaces (`use_separable=True`):** Enabling separable convolutions breaks typical 2D processing matrices into 1D depthwise and pointwise components. QKeras gives you fine-grained control to quantize every distinct subsection of this routing individually—allocating independent targets for `depthwise_quantizer`, `pointwise_quantizer`, and the internal tracking block accumulation buffer `acc_quantizer`.

### 1. Environment Setup & Dependency Installation
Uncomment the cell below if you need to install the required libraries within your current execution notebook instance.

In [1]:
# !pip install qkeras-v3 keras tensorflow

### 2. Imports, Custom Layer Serializer and Model Block Formations

In [2]:
import keras
from keras import initializers
from keras import ops as Kops
from keras.layers import Activation, Input, UpSampling2D
from keras.models import Model

from qkeras import *

def create_model():
    """Constructs a dual-frequency QOctaveConv2D structural network."""
    kernel_initializer = initializers.he_normal(seed=42)

    x = x_in = Input(shape=(256, 256, 3), name="main_input")

    # Block 1: Seeds dual-frequency pathways out from uniform raw image tensor space
    high, low = QOctaveConv2D(
        32,
        (3, 3),
        alpha=0.5,
        strides=(2, 2),
        padding="valid",
        kernel_initializer=kernel_initializer,
        bias_initializer="zeros",
        bias_quantizer="quantized_bits(4,1)",
        depthwise_quantizer="quantized_bits(4,1)",
        depthwise_activation="quantized_bits(6,2,1)",
        pointwise_quantizer="quantized_bits(4,1)",
        acc_quantizer="quantized_bits(16,7,1)",
        activation="quantized_relu(6,2)",
        use_separable=True,
        name="block1_conv1",
)([x, None])

    # Block 2
    high, low = QOctaveConv2D(
        64,
        (3, 3),
        alpha=0.4,
        strides=(2, 2),
        padding="same",
        kernel_initializer=kernel_initializer,
        bias_initializer="zeros",
        bias_quantizer="quantized_bits(4,1)",
        depthwise_quantizer="quantized_bits(4,1)",
        depthwise_activation="quantized_bits(6,2,1)",
        pointwise_quantizer="quantized_bits(4,1)",
        acc_quantizer="quantized_bits(16,7,1)",
        activation="quantized_relu(6,2)",
        use_separable=True,
        name="block2_conv1",
)([high, low])

    # Block 3
    high, low = QOctaveConv2D(
        64,
        (3, 3),
        alpha=0.4,
        strides=(2, 2),
        padding="same",
        kernel_initializer=kernel_initializer,
        bias_initializer="zeros",
        bias_quantizer="quantized_bits(4,1)",
        depthwise_quantizer="quantized_bits(4,1)",
        depthwise_activation="quantized_bits(6,2,1)",
        pointwise_quantizer="quantized_bits(4,1)",
        acc_quantizer="quantized_bits(16,7,1)",
        activation="quantized_relu(6,2)",
        use_separable=True,
        name="block3_conv1",
)([high, low])

    high, low = QOctaveConv2D(
        32,
        (3, 3),
        alpha=0.4,
        strides=(1, 1),
        padding="same",
        kernel_initializer=kernel_initializer,
        bias_initializer="zeros",
        bias_quantizer="quantized_bits(4,1)",
        depthwise_quantizer="quantized_bits(4,1)",
        depthwise_activation="quantized_bits(6,2,1)",
        pointwise_quantizer="quantized_bits(4,1)",
        acc_quantizer="quantized_bits(16,7,1)",
        activation="quantized_relu(6,2)",
        use_separable=True,
        name="block3_conv2",
)([high, low])

    high, low = QOctaveConv2D(
        32,
        (3, 3),
        alpha=0.3,
        strides=(1, 1),
        padding="same",
        kernel_initializer=kernel_initializer,
        bias_initializer="zeros",
        bias_quantizer="quantized_bits(4,1)",
        depthwise_quantizer="quantized_bits(4,1)",
        depthwise_activation="quantized_bits(6,2,1)",
        pointwise_quantizer="quantized_bits(4,1)",
        acc_quantizer="quantized_bits(16,7,1)",
        activation="quantized_relu(6,2)",
        use_separable=True,
        name="block3_conv3",
)([high, low])

    # Aggregates and collapses the frequency pathways back to alpha=0
    x, _ = QOctaveConv2D(
        32,
        (3, 3),
        alpha=0.0,
        strides=(2, 2),
        padding="same",
        kernel_initializer=kernel_initializer,
        bias_initializer="zeros",
        bias_quantizer="quantized_bits(4,1)",
        depthwise_quantizer="quantized_bits(4,1)",
        depthwise_activation="quantized_bits(6,2,1)",
        pointwise_quantizer="quantized_bits(4,1)",
        acc_quantizer="quantized_bits(16,7,1)",
        activation="quantized_relu(6,2)",
        use_separable=True,
        name="block3_conv_down",
)([high, low])

    # Upsample dimensional map
    x = UpSampling2D(size=(2, 2), data_format="channels_last", name="upsample_2d")(x)

    x = QConv2D(
        2,
        (2, 2),
        strides=(1, 1),
        kernel_initializer=kernel_initializer,
        bias_initializer="ones",
        kernel_quantizer=quantized_bits(4, 0, 1),
        bias_quantizer=quantized_bits(4, 0, 1),
        padding="same",
        name="conv_up",
)(x)

    x = Activation("softmax", name="softmax")(x)
    return Model(x_in, x, name="qoctave_network")

@keras.saving.register_keras_serializable()
def customLoss(y_true, y_pred):
    """Custom weighted log focal-style loss for edge segmentation stability."""
    log1 = 1.5 * y_true * Kops.log(y_pred + 1e-9) * keras.ops.power(1 - y_pred, 2)
    log0 = 0.5 * (1 - y_true) * Kops.log((1 - y_pred) + 1e-9) * keras.ops.power(y_pred, 2)
    return -Kops.sum(keras.ops.mean(log0 + log1, axis=0))

/Users/mariuskoppel/cms/qkeras/venv/lib/python3.11/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


### 3. Initialize Model Infrastructure and View Resource Cost Profile

In [3]:
model = create_model()
model.compile(optimizer="Adam", loss=customLoss, metrics=["acc"])

# Display comprehensive model metrics summary table
model.summary(line_length=100)

print("\n--- QKeras Internal Hardware Profiling Operational Statistics ---")
print_qstats(model)

Model: "qoctave_network"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                ┃ Output Shape            ┃        Param # ┃ Connected to            ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ main_input (InputLayer)     │ (None, 256, 256, 3)     │              0 │ -                       │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ block1_conv1_avg_h_to_l     │ (None, 128, 128, 3)     │              0 │ main_input[0][0]        │
│ (QAveragePooling2D)         │                         │                │                         │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ block1_conv1_c_h_to_l       │ (None, 63, 63, 16)      │             91 │ block1_conv1_avg_h_to_… │
│ (QSeparableConv2D)          │                         │                │                         │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ block1_conv1_c_h_to_h       │ (None, 127, 127, 16)    │             91 │ main_input[0][0]        │
│ (QSeparableConv2D)          │                         │                │                         │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ block1_conv1_c_h_to_l_act_… │ (None, 63, 63, 16)      │              0 │ block1_conv1_c_h_to_l[… │
│ (QActivation)               │                         │                │                         │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ block1_conv1_c_h_to_h_act_… │ (None, 127, 127, 16)    │              0 │ block1_conv1_c_h_to_h[… │
│ (QActivation)               │                         │                │                         │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ block1_conv1_l_act_qr62     │ (None, 63, 63, 16)      │              0 │ block1_conv1_c_h_to_l_… │
│ (QActivation)               │                         │                │                         │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ block1_conv1_h_act_qr62     │ (None, 127, 127, 16)    │              0 │ block1_conv1_c_h_to_h_… │
│ (QActivation)               │                         │                │                         │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ block2_conv1_c_l_to_h       │ (None, 32, 32, 38)      │            790 │ block1_conv1_l_act_qr6… │
│ (QSeparableConv2D)          │                         │                │                         │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ block2_conv1_c_h_to_h       │ (None, 64, 64, 38)      │            790 │ block1_conv1_h_act_qr6… │
│ (QSeparableConv2D)          │                         │                │                         │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ block2_conv1_c_l_to_h_act_… │ (None, 32, 32, 38)      │              0 │ block2_conv1_c_l_to_h[… │
│ (QActivation)               │                         │                │                         │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ block2_conv1_avg_h_to_l     │ (None, 63, 63, 16)      │              0 │ block1_conv1_h_act_qr6… │
│ (QAveragePooling2D)         │                         │                │                         │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ block2_conv1_c_h_to_h_act_… │ (None, 64, 64, 38)      │              0 │ block2_conv1_c_h_to_h[… │
│ (QActivation)               │                         │    

 Total params: 14,872 (58.09 KB)

 Trainable params: 14,872 (58.09 KB)

 Non-trainable params: 0 (0.00 B)


--- QKeras Internal Hardware Profiling Operational Statistics ---

Number of operations in model:
    block1_conv1_c_h_to_h         : 693547 (smult_4_8)
    block1_conv1_c_h_to_l         : 170667 (smult_4_8)
    block2_conv1_c_h_to_h         : 745472 (smult_4_6)
    block2_conv1_c_h_to_l         : 174080 (smult_4_6)
    block2_conv1_c_l_to_h         : 186368 (smult_4_6)
    block2_conv1_c_l_to_l         : 174080 (smult_4_6)
    block3_conv1_c_h_to_h         : 389120 (smult_4_6)
    block3_conv1_c_h_to_l         : 94208 (smult_4_6)
    block3_conv1_c_l_to_h         : 69632 (smult_4_6)
    block3_conv1_c_l_to_l         : 66560 (smult_4_6)
    block3_conv2_c_h_to_h         : 369664 (smult_4_6)
    block3_conv2_c_h_to_l         : 90880 (smult_4_6)
    block3_conv2_c_l_to_h         : 64768 (smult_4_6)
    block3_conv2_c_l_to_l         : 63232 (smult_4_6)
    block3_conv3_c_h_to_h         : 197632 (smult_4_6)
    block3_conv3_c_h_to_l         : 46336 (smult_4_6)
    block3_conv3_c_l_to_h   